# task_19 Solution

In [1]:

from pathlib import Path
import pandas as pd


base = Path("../../tasks/task_19/data/")


In [2]:

matches = pd.read_csv(base / "matches.csv")
events = pd.read_csv(base / "events.csv")

Q1: Using only first-half goals (minute <= 45), recompute match outcomes and standings with 3-1-0 points. Which team finishes top? Tie-break: GD, GF, alphabetical.

In [3]:
goals = events[events["event_type"] == "goal"]
first_half_goals = goals[goals["minute"] <= 45]

fh_by_match = first_half_goals.groupby(["match_id", "team_id"]).size().unstack(fill_value=0)
all_teams = set(matches["home_team"]) | set(matches["away_team"])

standings = {t: {"pts": 0, "gd": 0, "gf": 0} for t in all_teams}
for _, match in matches.iterrows():
    mid, home, away = match["match_id"], match["home_team"], match["away_team"]
    hg = fh_by_match.loc[mid, home] if mid in fh_by_match.index and home in fh_by_match.columns else 0
    ag = fh_by_match.loc[mid, away] if mid in fh_by_match.index and away in fh_by_match.columns else 0
    if hg > ag:
        standings[home]["pts"] += 3
    elif hg == ag:
        standings[home]["pts"] += 1
        standings[away]["pts"] += 1
    else:
        standings[away]["pts"] += 3
    standings[home]["gd"] += hg - ag
    standings[away]["gd"] += ag - hg
    standings[home]["gf"] += hg
    standings[away]["gf"] += ag

q1 = sorted(standings.keys(), key=lambda t: (-standings[t]["pts"], -standings[t]["gd"], -standings[t]["gf"], t))[0]
print(q1)

TA


Q3: In what percentage of matches did the team leading at halftime fail to win the match, rounded to 1 decimal place?

In [4]:
first_half_goals = goals[goals["minute"] <= 45]
halftime = first_half_goals.groupby(["match_id", "team_id"]).size().unstack(fill_value=0)
fulltime = goals.groupby(["match_id", "team_id"]).size().unstack(fill_value=0)
fail_count = 0
denom = 0
for row in matches.itertuples(index=False):
    h1 = halftime.loc[row.match_id, row.home_team] if row.match_id in halftime.index and row.home_team in halftime.columns else 0
    a1 = halftime.loc[row.match_id, row.away_team] if row.match_id in halftime.index and row.away_team in halftime.columns else 0
    if h1 == a1:
        continue
    denom += 1
    hg = fulltime.loc[row.match_id, row.home_team] if row.match_id in fulltime.index and row.home_team in fulltime.columns else 0
    ag = fulltime.loc[row.match_id, row.away_team] if row.match_id in fulltime.index and row.away_team in fulltime.columns else 0
    if h1 > a1 and not (hg > ag):
        fail_count += 1
    if a1 > h1 and not (ag > hg):
        fail_count += 1
q3 = f"{(100 * fail_count / denom):.1f}%"
print(q3)


11.1%
